# Qfind Ingestion

This notebook embeds the prepared CUAD starter evidence records and stores them in local Qdrant. Run the cells from top to bottom.

## Workflow

1. Load `starter_clause_evidence.jsonl`.
2. Connect to embedded local Qdrant at `../data/qdrant_local`.
3. Load the embedding model.
4. Embed each evidence record's `text` field.
5. Upsert vectors plus metadata into Qdrant.
6. Run a small search to confirm retrieval works.

## Imports and Settings

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer

CONTRACT_COLLECTION = "contracts_clause_evidence"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
DATA_PATH = Path("../data/processed/starter_clause_evidence.jsonl")
QDRANT_PATH = Path("../data/qdrant_local")

# Set this to False if you want to add/update records without deleting the collection first.
RECREATE_COLLECTION = True

c:\Users\User\OneDrive - National University of Singapore\Latest personal projects\Qfind\.conda-qfind\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Prepared Evidence Records

In [ ]:
records = []

with DATA_PATH.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} records")
records[0]

Loaded 463 records


{'id': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement::Anti-Assignment::1',
 'document_id': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement',
 'source_pdf': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement.pdf',
 'source_txt': 'data\\cuad\\CUAD_v1\\full_contract_txt\\Part_II\\TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement.txt',
 'clause_type': 'Anti-Assignment',
 'answer': 'Yes',
 'text': 'Notwithstanding the foregoing, upon the prior written approval of Online BVI, which approval may be withheld in its sole discretion, the Company shall be permitted to sublicense its rights hereunder to a wholly-owned Subsidiary of the Company or a majority-owned Subsidiary of Tom Holding, for the same purpose and under the same terms and conditions as the license set forth herein.'}

## Connect to Qdrant and Load the Embedding Model

In [ ]:
# Embedded Qdrant stores vectors in a local folder, so Docker is not needed.
client = QdrantClient(path=str(QDRANT_PATH))

# Loading the model can take a little while the first time.
model = SentenceTransformer(EMBEDDING_MODEL)

if RECREATE_COLLECTION and client.collection_exists(collection_name=CONTRACT_COLLECTION):
    client.delete_collection(collection_name=CONTRACT_COLLECTION)

if not client.collection_exists(collection_name=CONTRACT_COLLECTION):
    client.create_collection(
        collection_name=CONTRACT_COLLECTION,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )

print(f"Ready: {CONTRACT_COLLECTION}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2953.51it/s]


Ready: contracts_clause_evidence


## Create Stable Qdrant IDs

Turn a text ID into a stable integer ID for Qdrant.

In [ ]:
def stable_point_id(raw_id: str) -> int:
    """Turn a text ID into a stable integer ID for Qdrant."""
    digest = hashlib.sha256(raw_id.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], byteorder="big", signed=False)


stable_point_id(records[0]["id"])

17069622490215727222

## Embed and Upsert Records

In [ ]:
texts = [record["text"] for record in records]

#process 32 texts at a time , turn text string into vector numbbers
embeddings = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)
#builds qdrant points
points = [
    PointStruct(
        id=stable_point_id(record["id"]),
        vector=embedding.tolist(),
        payload=record,
    )
    for record, embedding in zip(records, embeddings)
]

client.upsert(collection_name=CONTRACT_COLLECTION, points=points)

count = client.count(collection_name=CONTRACT_COLLECTION, exact=True).count
print(f"Upserted {len(points)} records")
print(f"Collection count: {count}")

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches: 100%|██████████| 15/15 [00:23<00:00,  1.59s/it]


Upserted 463 records
Collection count: 926


## Sanity Search

In [ ]:
query = "Does the contract restrict assignment?"
query_vector = model.encode(query, normalize_embeddings=True).tolist()

results = client.query_points(
    collection_name=CONTRACT_COLLECTION,
    query=query_vector,
    limit=5,
    with_payload=True,
)

for index, point in enumerate(results.points, start=1):
    payload = point.payload
    print(f"Result {index}: score={point.score:.3f}")
    print(f"Clause type: {payload['clause_type']}")
    print(f"Source: {payload['source_pdf']}")
    print(payload["text"][:500])
    print("-" * 80)

Result 1: score=0.799
Clause type: Anti-Assignment
Source: DovaPharmaceuticalsInc_20181108_10-Q_EX-10.2_11414857_EX-10.2_Promotion Agreement.pdf
Except as provided in this Section 13.2, this Agreement may not be assigned or otherwise transferred, nor may any rights or obligations hereunder be assigned or transferred, by either Party, without the written consent of the other Party (such consent not to be unreasonably withheld); provided that a merger, sale of stock or comparable transaction shall not constitute an assignment.
--------------------------------------------------------------------------------
Result 2: score=0.799
Clause type: Anti-Assignment
Source: DovaPharmaceuticalsInc_20181108_10-Q_EX-10.2_11414857_EX-10.2_Promotion Agreement.pdf
Except as provided in this Section 13.2, this Agreement may not be assigned or otherwise transferred, nor may any rights or obligations hereunder be assigned or transferred, by either Party, without the written consent of the other Party (such